In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros con la distribución de población por edad y sección censal

Se han estructurado los datos en la carpeta de inputs para la dimensión demográfica de forma que se dispone de un fichero CSV a nivel
de provincia. Así pues, se han definido una función en las utilidades que se encargan de cargar y dar una limpieza inicial a los datos.

Los datos se obtienen del censo anual de población
https://www.ine.es/dynt3/inebase/index.htm?padre=11555&capsel=11100

In [2]:
path = os.path.join(DATA_INPUTS_DD, "Poblacion por sexo y edad")

indicadores = carga_datos_ine(path)

# Veo una muestra de su estructura y contenido
print(indicadores.info())
indicadores.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 936408 entries, 528 to 1532255
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Provincias  936408 non-null  object 
 1   Municipios  936408 non-null  object 
 2   Secciones   936408 non-null  object 
 3   Sexo        936408 non-null  object 
 4   Edad        936408 non-null  object 
 5   Periodo     936408 non-null  int64  
 6   Total       931854 non-null  float64
 7   Provincia   936408 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 64.3+ MB
None


,Provincias,Municipios,Secciones,Sexo,Edad,Periodo,Total,Provincia
1182031,47 Valladolid,47012 Bahabón,4701201001 Bahabón sección 01001,Hombres,De 10 a 14 años,2021,1.0,Valladolid
723607,37 Salamanca,37042 Barceo,3704201001 Barceo sección 01001,Mujeres,De 80 a 84 años,2021,0.0,Salamanca
1406080,49 Zamora,49039 Casaseca de las Chanas,4903901001 Casaseca de las Chanas sección 01001,Total,De 15 a 19 años,2024,9.0,Zamora
1013804,40 Segovia,40155 Palazuelos de Eresma,4015501001 Palazuelos de Eresma sección 01001,Total,De 50 a 54 años,2024,323.0,Segovia
985502,40 Segovia,40092 Fuentidueña,4009201001 Fuentidueña sección 01001,Mujeres,De 90 a 94 años,2022,3.0,Segovia


# Filtrado
Interesan ahora la fragmentación de la población por grupos quinquenales, independientemente del sexo

In [3]:
indicadores_filtrado = indicadores[
    (indicadores["Sexo"] == "Total") &
    (indicadores["Edad"] != "Todas las edades")
].copy()

# Veo una muestra de su estructura y contenido
print(indicadores_filtrado.info())
indicadores_filtrado.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 297948 entries, 532 to 1532079
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Provincias  297948 non-null  object 
 1   Municipios  297948 non-null  object 
 2   Secciones   297948 non-null  object 
 3   Sexo        297948 non-null  object 
 4   Edad        297948 non-null  object 
 5   Periodo     297948 non-null  int64  
 6   Total       296499 non-null  float64
 7   Provincia   297948 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 20.5+ MB
None


,Provincias,Municipios,Secciones,Sexo,Edad,Periodo,Total,Provincia
1359886,47 Valladolid,47190 Velilla,4719001001 Velilla sección 01001,Total,De 20 a 24 años,2022,4.0,Valladolid
127787,05 Ávila,"05232 Serrada, La","0523201001 Serrada, La sección 01001",Total,De 5 a 9 años,2021,6.0,Avila
195917,09 Burgos,09059 Burgos,0905905039 Burgos sección 05039,Total,De 30 a 34 años,2023,38.0,Burgos
742143,37 Salamanca,37078 Candelario,3707801001 Candelario sección 01001,Total,De 40 a 44 años,2021,58.0,Salamanca
348234,09 Burgos,09361 Sargentes de la Lora,0936101001 Sargentes de la Lora sección 01001,Total,De 15 a 19 años,2022,2.0,Burgos


# Estandarización del dataframe de datos del INE

Como se puede observar, el fichero csv de datos del INE tiene un formato poco amigable para el tratamiento de los datos. En lugar de tener una fila
por cada par sección-año y varias columnas (una por factor), tiene múltiples filas con distintos indicadores para una misma sección, lo que resulta
complejo de tratar. Además, se observa como se mezcla el código del municipio, distrito y seccion con el texto, y deberían tener una columna con
los códigos.

In [4]:
indicadores_estandarizados = estandarizar_df_ine(indicadores_filtrado, "Edad")
print(indicadores_estandarizados.info())
indicadores_estandarizados.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 297948 entries, 532 to 1532079
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Provincia  297948 non-null  object 
 1   CMuni      297948 non-null  object 
 2   CUSEC      297948 non-null  object 
 3   Indicador  297948 non-null  object 
 4   Periodo    297948 non-null  int64  
 5   Total      296499 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 15.9+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
496136,Leon,24115,2411501008,De 95 a 99 años,2024,7.0
634924,Palencia,34120,3412004005,De 0 a 4 años,2024,80.0
879455,Salamanca,37278,3727801001,De 80 a 84 años,2021,15.0
26949,Avila,05035,0503501001,De 20 a 24 años,2023,6.0
1003500,Segovia,40132,4013201001,De 40 a 44 años,2024,1.0


In [5]:
# Modifico el tipo de Total a "Entero"
indicadores_estandarizados["Total"] = pd.to_numeric(
    indicadores_estandarizados["Total"], errors="coerce"
).astype("Int64")  # <-- con mayúscula, permite NaN

In [6]:
# Pivoto los indicadores para tener una columna por indicador y reducir las filas de la tabla
indicadores_por_seccion = pivotar_indicadores(indicadores_estandarizados)
print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14119 entries, 0 to 14118
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Provincia        14119 non-null  object
 1   CMuni            14119 non-null  object
 2   CUSEC            14119 non-null  object
 3   Periodo          14119 non-null  int64 
 4   100_y_más_años   14119 non-null  Int64 
 5   De_0_a_4_años    14119 non-null  Int64 
 6   De_10_a_14_años  14119 non-null  Int64 
 7   De_15_a_19_años  14119 non-null  Int64 
 8   De_20_a_24_años  14119 non-null  Int64 
 9   De_25_a_29_años  14119 non-null  Int64 
 10  De_30_a_34_años  14119 non-null  Int64 
 11  De_35_a_39_años  14119 non-null  Int64 
 12  De_40_a_44_años  14119 non-null  Int64 
 13  De_45_a_49_años  14119 non-null  Int64 
 14  De_5_a_9_años    14119 non-null  Int64 
 15  De_50_a_54_años  14119 non-null  Int64 
 16  De_55_a_59_años  14119 non-null  Int64 
 17  De_60_a_64_años  14119 non-null

,Provincia,CMuni,CUSEC,Periodo,100_y_más_años,De_0_a_4_años,De_10_a_14_años,De_15_a_19_años,De_20_a_24_años,De_25_a_29_años,...,De_50_a_54_años,De_55_a_59_años,De_60_a_64_años,De_65_a_69_años,De_70_a_74_años,De_75_a_79_años,De_80_a_84_años,De_85_a_89_años,De_90_a_94_años,De_95_a_99_años
9858,Soria,42036,4203601001,2023,0,0,0,0,1,1,...,5,4,5,5,4,2,3,2,3,2
7488,Salamanca,37240,3724001001,2022,1,54,53,62,61,64,...,119,114,83,67,36,35,27,17,14,7
7645,Salamanca,37274,3727401003,2023,0,22,25,40,39,43,...,46,64,49,56,52,51,30,56,27,12
7531,Salamanca,37248,3724801001,2021,0,0,2,6,9,11,...,13,18,13,14,6,11,7,8,6,1
264,Avila,05024,0502401001,2021,3,2,1,1,4,2,...,18,20,13,15,13,14,18,20,6,5


## Filtrado de años
Revisando la documentación del INE, en el año 2021 se cambió radicalemente la metodología que define las secciones censales,
y en concreto en Castilla y León se aumentó el numero de censos de 2700 a unos 3500 apróximadamente. Es por ello que, si bien
se dispone de datos de años anteriores, sería complejo y peligroso fragmentar y proyectar los censos de años previos en la malla 
censal actual, por lo que se filtraran datos de años previos

In [7]:
# # Filtro por los años 2021 - 2023.
indicadores_por_seccion = indicadores_por_seccion[
    indicadores_por_seccion["Periodo"].isin([2021, 2022, 2023])
].copy()

# Agrupación por tramos de edad

Aquí lo interesante de cara a tasas de envejecimiento y dependencia es agrupar por los 3 tramos de edad siguientes:

1. Menores de 16 años
2. Entre 16 y 65 años
3. Mayores de 65 años

Es necesario agrupar estos datos y reducir sustanciamente las columnas. Lo que sucede es que no se dispone empíricamente
del dato para cortar a los 16 años. Sin embargo, se asumirá metodológicamente una distribución uniforme de población entre
los 15 y los 19 años de edad, obteniendo que de 15 a 16 años es un 20% de esa muestra aproximadamente

In [8]:
# --- Definición de grupos etarios ---
col_0_4 = "De_0_a_4_años"
col_5_9 = "De_5_a_9_años"
col_10_14 = "De_10_a_14_años"
col_15_19 = "De_15_a_19_años"

col_20_64 = [
    "De_20_a_24_años", "De_25_a_29_años", "De_30_a_34_años",
    "De_35_a_39_años", "De_40_a_44_años", "De_45_a_49_años",
    "De_50_a_54_años", "De_55_a_59_años", "De_60_a_64_años"
]

col_65_plus = [
    "De_65_a_69_años", "De_70_a_74_años", "De_75_a_79_años",
    "De_80_a_84_años", "De_85_a_89_años", "De_90_a_94_años",
    "De_95_a_99_años", "100_y_más_años"
]

# --- Cálculo ponderado ---
indicadores_por_seccion["poblacion__menor_16"] = (
    indicadores_por_seccion[col_0_4]
    + indicadores_por_seccion[col_5_9]
    + indicadores_por_seccion[col_10_14]
    + 0.2 * indicadores_por_seccion[col_15_19]
).round().astype("Int64")

indicadores_por_seccion["poblacion_16_64"] = (
    0.8 * indicadores_por_seccion[col_15_19]
    + indicadores_por_seccion[col_20_64].sum(axis=1)
).round().astype("Int64")

indicadores_por_seccion["poblacion_mayor_65"] = indicadores_por_seccion[col_65_plus].sum(axis=1)

print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 10597 entries, 0 to 14117
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Provincia            10597 non-null  object
 1   CMuni                10597 non-null  object
 2   CUSEC                10597 non-null  object
 3   Periodo              10597 non-null  int64 
 4   100_y_más_años       10597 non-null  Int64 
 5   De_0_a_4_años        10597 non-null  Int64 
 6   De_10_a_14_años      10597 non-null  Int64 
 7   De_15_a_19_años      10597 non-null  Int64 
 8   De_20_a_24_años      10597 non-null  Int64 
 9   De_25_a_29_años      10597 non-null  Int64 
 10  De_30_a_34_años      10597 non-null  Int64 
 11  De_35_a_39_años      10597 non-null  Int64 
 12  De_40_a_44_años      10597 non-null  Int64 
 13  De_45_a_49_años      10597 non-null  Int64 
 14  De_5_a_9_años        10597 non-null  Int64 
 15  De_50_a_54_años      10597 non-null  Int64 
 16  De_55_a_5

,Provincia,CMuni,CUSEC,Periodo,100_y_más_años,De_0_a_4_años,De_10_a_14_años,De_15_a_19_años,De_20_a_24_años,De_25_a_29_años,...,De_65_a_69_años,De_70_a_74_años,De_75_a_79_años,De_80_a_84_años,De_85_a_89_años,De_90_a_94_años,De_95_a_99_años,poblacion__menor_16,poblacion_16_64,poblacion_mayor_65
4887,Leon,24142,2414202013,2023,0,44,69,76,70,45,...,70,30,30,24,17,3,0,182,954,174
10447,Soria,42183,4218301001,2023,0,0,0,0,4,1,...,12,8,14,10,5,3,1,0,43,53
9781,Soria,42020,4202001001,2022,1,49,74,84,87,88,...,86,87,69,38,67,27,16,201,962,391
2174,Burgos,09073,0907301001,2021,0,67,119,80,50,38,...,40,36,24,13,10,7,2,315,970,132
8828,Segovia,40059,4005901001,2021,0,2,2,9,9,8,...,5,3,5,8,6,5,2,6,89,34


# Cálculo de las tasas de dependencia y envejecimiento

## Fórmulas

**Índice de envejecimiento**

$Índice\ de\ Envejecimiento = \dfrac{Población\ de\ 65\ y\ más\ años}{Población\ menor\ de\ 16\ años}$

**Tasa de dependencia total**

$Tasa\ de\ Dependencia = \dfrac{Población\ menor\ de\ 16\ años + Población\ de\ 65\ y\ más\ años}{Población\ de\ 16\ a\ 64\ años}$

## Importancia

Estos indicadores permiten analizar el **equilibrio demográfico y la sostenibilidad social** de un territorio.  
- El **índice de envejecimiento** muestra el grado de envejecimiento de la población y su impacto potencial en los servicios sanitarios y sociales.  
- La **tasa de dependencia** mide la proporción de población potencialmente dependiente respecto a la población activa, siendo esencial para planificar políticas de empleo, cuidados y protección social.


In [9]:
# Cálculo de tasas de envejecimiento y de dependencia
indicadores_por_seccion["Indice_Envejecimiento"] = (
    indicadores_por_seccion["poblacion_mayor_65"] /
    indicadores_por_seccion["poblacion__menor_16"]
).round(2)

indicadores_por_seccion["Tasa_Dependencia"] = (
    (indicadores_por_seccion["poblacion__menor_16"] + indicadores_por_seccion["poblacion_mayor_65"]) /
    indicadores_por_seccion["poblacion_16_64"]
).round(2)

indicadores_por_seccion.replace([np.inf, -np.inf], np.nan, inplace=True)

print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 10597 entries, 0 to 14117
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Provincia              10597 non-null  object 
 1   CMuni                  10597 non-null  object 
 2   CUSEC                  10597 non-null  object 
 3   Periodo                10597 non-null  int64  
 4   100_y_más_años         10597 non-null  Int64  
 5   De_0_a_4_años          10597 non-null  Int64  
 6   De_10_a_14_años        10597 non-null  Int64  
 7   De_15_a_19_años        10597 non-null  Int64  
 8   De_20_a_24_años        10597 non-null  Int64  
 9   De_25_a_29_años        10597 non-null  Int64  
 10  De_30_a_34_años        10597 non-null  Int64  
 11  De_35_a_39_años        10597 non-null  Int64  
 12  De_40_a_44_años        10597 non-null  Int64  
 13  De_45_a_49_años        10597 non-null  Int64  
 14  De_5_a_9_años          10597 non-null  Int64  
 15  De_50_a

,Provincia,CMuni,CUSEC,Periodo,100_y_más_años,De_0_a_4_años,De_10_a_14_años,De_15_a_19_años,De_20_a_24_años,De_25_a_29_años,...,De_75_a_79_años,De_80_a_84_años,De_85_a_89_años,De_90_a_94_años,De_95_a_99_años,poblacion__menor_16,poblacion_16_64,poblacion_mayor_65,Indice_Envejecimiento,Tasa_Dependencia
9446,Segovia,40194,4019404003,2023,1,29,34,52,50,53,...,54,44,34,17,3,116,548,258,2.22,0.68
3201,Burgos,09374,0937401001,2022,0,1,6,5,7,9,...,6,6,1,1,1,10,78,28,2.8,0.49
6620,Salamanca,37035,3703501001,2021,0,0,5,21,9,6,...,14,9,8,7,1,13,119,63,4.85,0.64
5526,Palencia,34039,3403901001,2022,0,0,0,1,1,0,...,7,5,7,1,0,0,23,32,<NA>,1.39
524,Avila,05090,0509001001,2023,0,0,0,0,1,4,...,9,3,3,2,0,1,33,28,28.0,0.88


# Agregado de datos

Aunque para el modelo y el detalle fino se dispondrá de los datos desagregados, es interesante también disponer de los datos
agregados a 2 niveles para la visualizacion:

1. Nivel municipal (por CMuni)
2. Nivel provincial (por Provinincia)


In [10]:
# Aseguramos que los campos numéricos estén correctamente convertidos
columnas_enteras = [
    '100_y_más_años', 'De_0_a_4_años', 'De_5_a_9_años', 'De_10_a_14_años', 'De_15_a_19_años',
    'De_20_a_24_años', 'De_25_a_29_años', 'De_30_a_34_años', 'De_35_a_39_años', 'De_40_a_44_años',
    'De_45_a_49_años', 'De_50_a_54_años', 'De_55_a_59_años', 'De_60_a_64_años', 'De_65_a_69_años',
    'De_70_a_74_años', 'De_75_a_79_años', 'De_80_a_84_años', 'De_85_a_89_años', 'De_90_a_94_años',
    'De_95_a_99_años', 'poblacion__menor_16', 'poblacion_16_64', 'poblacion_mayor_65'
]

# --- 1️⃣ NIVEL MUNICIPAL ---
indicadores_por_municipio = (
    indicadores_por_seccion
    .groupby(["Provincia", "CMuni", "Periodo"], as_index=False)
    .agg({
        **{col: "sum" for col in columnas_enteras},
    })
)

# Las tasas deben ponderarse por la población del municipio sino saldrían valores irreales
indicadores_por_municipio["Indice_Envejecimiento"] = (
    indicadores_por_municipio["poblacion_mayor_65"] /
    indicadores_por_municipio["poblacion__menor_16"]
)
indicadores_por_municipio["Tasa_Dependencia"] = (
    (indicadores_por_municipio["poblacion__menor_16"] +
     indicadores_por_municipio["poblacion_mayor_65"]) /
    indicadores_por_municipio["poblacion_16_64"]
)

# --- 2️⃣ NIVEL PROVINCIAL ---
indicadores_por_provincia = (
    indicadores_por_seccion
    .groupby(["Provincia", "Periodo"], as_index=False)
    .agg({
        **{col: "sum" for col in columnas_enteras},
    })
)

# Las tasas deben ponderarse por la población del municipio sino saldrían valores irreales
indicadores_por_provincia["Indice_Envejecimiento"] = (
    indicadores_por_provincia["poblacion_mayor_65"] /
    indicadores_por_provincia["poblacion__menor_16"]
)

indicadores_por_provincia["Tasa_Dependencia"] = (
    (indicadores_por_provincia["poblacion__menor_16"] +
     indicadores_por_provincia["poblacion_mayor_65"]) /
    indicadores_por_provincia["poblacion_16_64"]
)

In [11]:
# Muestra
print(indicadores_por_municipio.info())
indicadores_por_municipio.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6744 entries, 0 to 6743
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Provincia              6744 non-null   object 
 1   CMuni                  6744 non-null   object 
 2   Periodo                6744 non-null   int64  
 3   100_y_más_años         6744 non-null   Int64  
 4   De_0_a_4_años          6744 non-null   Int64  
 5   De_5_a_9_años          6744 non-null   Int64  
 6   De_10_a_14_años        6744 non-null   Int64  
 7   De_15_a_19_años        6744 non-null   Int64  
 8   De_20_a_24_años        6744 non-null   Int64  
 9   De_25_a_29_años        6744 non-null   Int64  
 10  De_30_a_34_años        6744 non-null   Int64  
 11  De_35_a_39_años        6744 non-null   Int64  
 12  De_40_a_44_años        6744 non-null   Int64  
 13  De_45_a_49_años        6744 non-null   Int64  
 14  De_50_a_54_años        6744 non-null   Int64  
 15  De_5

,Provincia,CMuni,Periodo,100_y_más_años,De_0_a_4_años,De_5_a_9_años,De_10_a_14_años,De_15_a_19_años,De_20_a_24_años,De_25_a_29_años,...,De_75_a_79_años,De_80_a_84_años,De_85_a_89_años,De_90_a_94_años,De_95_a_99_años,poblacion__menor_16,poblacion_16_64,poblacion_mayor_65,Indice_Envejecimiento,Tasa_Dependencia
5702,Valladolid,47130,2023,0,0,2,3,1,3,3,...,9,2,7,2,0,5,62,33,6.6,0.612903
1173,Burgos,09183,2021,0,0,0,0,0,0,3,...,5,2,4,2,0,0,14,28,inf,2.0
503,Avila,05186,2023,6,31,38,67,84,64,62,...,112,93,70,53,18,153,973,553,3.614379,0.725591
609,Avila,05225,2021,0,2,6,9,17,12,13,...,28,25,26,18,3,20,208,145,7.25,0.793269
5059,Soria,42110,2022,0,1,0,1,2,3,2,...,4,5,1,0,0,2,47,20,10.0,0.468085


In [12]:
# Muestra
print(indicadores_por_provincia.info())
indicadores_por_provincia.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Provincia              27 non-null     object 
 1   Periodo                27 non-null     int64  
 2   100_y_más_años         27 non-null     Int64  
 3   De_0_a_4_años          27 non-null     Int64  
 4   De_5_a_9_años          27 non-null     Int64  
 5   De_10_a_14_años        27 non-null     Int64  
 6   De_15_a_19_años        27 non-null     Int64  
 7   De_20_a_24_años        27 non-null     Int64  
 8   De_25_a_29_años        27 non-null     Int64  
 9   De_30_a_34_años        27 non-null     Int64  
 10  De_35_a_39_años        27 non-null     Int64  
 11  De_40_a_44_años        27 non-null     Int64  
 12  De_45_a_49_años        27 non-null     Int64  
 13  De_50_a_54_años        27 non-null     Int64  
 14  De_55_a_59_años        27 non-null     Int64  
 15  De_60_a_

,Provincia,Periodo,100_y_más_años,De_0_a_4_años,De_5_a_9_años,De_10_a_14_años,De_15_a_19_años,De_20_a_24_años,De_25_a_29_años,De_30_a_34_años,...,De_75_a_79_años,De_80_a_84_años,De_85_a_89_años,De_90_a_94_años,De_95_a_99_años,poblacion__menor_16,poblacion_16_64,poblacion_mayor_65,Indice_Envejecimiento,Tasa_Dependencia
4,Burgos,2022,211,11769,15089,17014,16196,16215,15710,17522,...,16353,11396,10341,5747,1729,47102,220224,87520,1.858095,0.611296
17,Segovia,2023,118,5297,6510,7510,7639,7684,7765,8110,...,6408,4771,4515,2561,672,20833,98594,35905,1.723468,0.575471
19,Soria,2022,83,3068,3507,3949,4017,4262,4208,4398,...,4068,3149,3207,1796,578,11317,54465,22548,1.992401,0.621775
26,Zamora,2023,157,4070,5077,5956,6283,6650,6692,7438,...,9715,7819,7352,3942,1157,16348,97303,53276,3.25887,0.715538
0,Avila,2021,83,5168,6399,7175,7027,7208,7286,7890,...,7505,6190,5458,2862,797,20139,97574,41185,2.045037,0.628487


# Export de los resultados

In [13]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DD, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DD, "indicadores_edades_por_seccion.csv")
ruta_municipio = os.path.join(DATA_OUTPUTS_DD, "indicadores_edades_por_municipio.csv")
ruta_provincia = os.path.join(DATA_OUTPUTS_DD, "indicadores_edades_por_provincia.csv")

# Guardar DataFrames
indicadores_por_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
indicadores_por_municipio.to_csv(
    ruta_municipio,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
indicadores_por_provincia.to_csv(
    ruta_provincia,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DD}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DD_Dim_demografica
